# Notebook 07 – Exploratory Data Analysis (EDA)

This notebook explores the audited master dataset separately by source. It creates descriptive statistics, distributions, process–response plots, within-source correlations and a downloadable report archive.

**Input:** upload `master_experimental_dataset_qc.csv` from the Notebook 06 output ZIP. Do not use the unaudited master for this notebook.

EDA is descriptive. Correlations are not causal evidence, and separate studies must not be pooled simply because they share a column name.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from IPython.display import display
import shutil

uploaded = files.upload()
matches = [n for n in uploaded if n.lower().endswith('.csv') and 'master_experimental_dataset_qc' in n.lower()]
if not matches:
    raise FileNotFoundError('Upload master_experimental_dataset_qc.csv from the Notebook 06 outputs.')
df = pd.read_csv(matches[0])
print('Loaded:', matches[0], '| Shape:', df.shape)
if 'source_id' not in df.columns:
    raise ValueError('The audited master file must contain source_id.')
display(df.groupby('source_id').size().rename('records').to_frame())
display(df.head())

## 1. Define source-specific inputs and outcomes

The mapping follows the current audited master schema. Missing columns are skipped rather than guessed.

In [ ]:
SOURCE_MAP = {
 'conventional_laser_welding_dataset': {'inputs':['laser_power_W','welding_speed_mm_s'], 'targets':['weld_bead_width_mm','interface_width_mm','penetration_depth_mm'], 'eligibility':{'weld_bead_width_mm':'eligible_weld_bead_width','interface_width_mm':'eligible_interface_width','penetration_depth_mm':'eligible_penetration_depth_conventional'}},
 'alcu_wobbling_welding_dataset': {'inputs':['laser_power_W','tangential_wobbling_speed_mm_s','wobbling_diameter_mm'], 'targets':['bead_width_face_mm','bead_width_interface_mm','penetration_depth_mm'], 'eligibility':{'bead_width_face_mm':'eligible_bead_width_face_wobble','bead_width_interface_mm':'eligible_bead_width_interface_wobble','penetration_depth_mm':'eligible_penetration_depth_wobble'}},
 'shin_2021_metals_11050831': {'inputs':['laser_power_W','welding_speed_mm_s','reported_heat_input_J_mm'], 'targets':['tensile_shear_strength_MPa','electrical_resistance_microohm','welding_area_mm2'], 'eligibility':{'tensile_shear_strength_MPa':'eligible_tensile_shear_strength','electrical_resistance_microohm':'eligible_electrical_resistance','welding_area_mm2':'eligible_welding_area'}},
 'lee_2022_metals_12111957': {'inputs':['welding_speed_mm_s'], 'targets':['force_average_N_reported'], 'eligibility':{'force_average_N_reported':'eligible_force_average_N'}}
}
for sid in df['source_id'].dropna().unique():
    if sid not in SOURCE_MAP:
        print('No explicit variable map for source:', sid)


## 2. Source-specific descriptive statistics

Summaries are calculated only for numeric columns with observations. They are not pooled across source groups.

In [ ]:
outdir = Path('notebook07_eda_outputs')
figdir = outdir/'figures'
figdir.mkdir(parents=True, exist_ok=True)
summary_frames = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)].copy()
    if g.empty: continue
    cols = [c for c in cfg['inputs'] + cfg['targets'] if c in g.columns]
    numeric = g[cols].apply(pd.to_numeric, errors='coerce')
    desc = numeric.describe().T.reset_index().rename(columns={'index':'variable'})
    desc.insert(0, 'source_id', sid)
    summary_frames.append(desc)
    print(f'\\n{sid}: {len(g)} records')
    display(desc)
summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary.to_csv(outdir/'descriptive_statistics_by_source.csv', index=False)


## 3. Distribution plots by source

Each variable is plotted only within its own source group. Missing values remain in the data and are excluded only from the histogram.

In [ ]:
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)]
    if g.empty: continue
    for col in cfg['inputs'] + cfg['targets']:
        if col not in g.columns: continue
        vals = pd.to_numeric(g[col], errors='coerce').dropna()
        if vals.empty: continue
        fig, ax = plt.subplots(figsize=(6,4))
        ax.hist(vals, bins=min(10, max(3, int(np.sqrt(len(vals))))), edgecolor='black')
        ax.set_title(f'{sid}\\n{col}')
        ax.set_xlabel(col); ax.set_ylabel('Record count')
        fig.tight_layout()
        safe = ''.join(ch if ch.isalnum() else '_' for ch in f'{sid}_{col}')
        fig.savefig(figdir/f'distribution_{safe}.png', dpi=160)
        plt.show(); plt.close(fig)


## 4. Process–response scatter plots

Relationships are plotted within each source. If a target eligibility flag exists, only eligible rows are shown.

In [ ]:
scatter_rows = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)].copy()
    if g.empty: continue
    for xcol in cfg['inputs']:
        for ycol in cfg['targets']:
            if xcol not in g.columns or ycol not in g.columns: continue
            flag = cfg.get('eligibility', {}).get(ycol)
            plot_data = g.copy()
            if flag and flag in plot_data.columns:
                plot_data = plot_data.loc[plot_data[flag].fillna(False).astype(bool)]
            x = pd.to_numeric(plot_data[xcol], errors='coerce')
            y = pd.to_numeric(plot_data[ycol], errors='coerce')
            valid = x.notna() & y.notna()
            if valid.sum() < 2: continue
            fig, ax = plt.subplots(figsize=(6,4))
            ax.scatter(x[valid], y[valid])
            ax.set_title(f'{sid}\\n{ycol} vs {xcol} (n={int(valid.sum())})')
            ax.set_xlabel(xcol); ax.set_ylabel(ycol)
            fig.tight_layout()
            safe = ''.join(ch if ch.isalnum() else '_' for ch in f'{sid}_{ycol}_vs_{xcol}')
            fig.savefig(figdir/f'scatter_{safe}.png', dpi=160)
            plt.show(); plt.close(fig)
            scatter_rows.append({'source_id':sid,'input':xcol,'target':ycol,'n_plotted':int(valid.sum()),'eligibility_flag':flag or 'not_available'})
pd.DataFrame(scatter_rows).to_csv(outdir/'scatter_plot_inventory.csv', index=False)


## 5. Within-source correlation analysis

Pearson and Spearman coefficients are calculated separately within each source. Small samples and repeated process conditions can make estimates unstable. These are not causal effects.

In [ ]:
corr_rows = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)].copy()
    cols = [c for c in cfg['inputs'] + cfg['targets'] if c in g.columns]
    if len(g) < 3 or len(cols) < 2: continue
    numeric = g[cols].apply(pd.to_numeric, errors='coerce')
    for method in ['pearson','spearman']:
        corr = numeric.corr(method=method, min_periods=3)
        for i, a in enumerate(corr.columns):
            for b in corr.columns[i+1:]:
                if pd.notna(corr.loc[a,b]):
                    n_pair = int(numeric[[a,b]].dropna().shape[0])
                    corr_rows.append({'source_id':sid,'method':method,'variable_1':a,'variable_2':b,'correlation':corr.loc[a,b],'n_pairwise':n_pair})
corr_table = pd.DataFrame(corr_rows)
corr_table.to_csv(outdir/'correlations_within_source.csv', index=False)
display(corr_table)


## 6. Repeated-condition and quality-flag review

Repeated conditions are part of experimental design and are not automatically duplicates.

In [ ]:
repeat_summaries = []
grouping = {
 'conventional_laser_welding_dataset':['laser_power_W','welding_speed_mm_s'],
 'alcu_wobbling_welding_dataset':['laser_power_W','tangential_wobbling_speed_mm_s','wobbling_diameter_mm'],
 'shin_2021_metals_11050831':['laser_power_W','welding_speed_mm_s'],
 'lee_2022_metals_12111957':['upper_material','welding_speed_mm_s']
}
for sid, keys in grouping.items():
    g = df.loc[df['source_id'].eq(sid)]
    keys = [k for k in keys if k in g.columns]
    if not keys or g.empty: continue
    counts = g.groupby(keys, dropna=False).size().rename('records_at_condition').reset_index()
    counts.insert(0,'source_id',sid)
    repeat_summaries.append(counts)
repeat_table = pd.concat(repeat_summaries, ignore_index=True) if repeat_summaries else pd.DataFrame()
repeat_table.to_csv(outdir/'records_per_reported_condition.csv', index=False)
display(repeat_table)

flag_cols = [c for c in df.columns if c.endswith('_flag') or c.startswith('eligible_')]
if flag_cols:
    flag_summary = df.groupby('source_id')[flag_cols].agg(lambda s: s.fillna(False).astype(bool).sum())
    flag_summary.to_csv(outdir/'quality_and_eligibility_flag_counts.csv')
    display(flag_summary)


## 7. Export EDA tables and figures

The ZIP contains descriptive tables, correlations, repeat-condition summaries, flag counts and generated figures. The audited input file is not modified.

In [ ]:
(outdir/'README.txt').write_text('All summaries and plots are source-specific and exploratory. Correlations are not causal. Review source provenance and target definitions before modelling.\\n', encoding='utf-8')
zip_path = shutil.make_archive('notebook07_eda_outputs', 'zip', outdir)
print('Created:', zip_path)
files.download(zip_path)
